<a href="https://colab.research.google.com/github/AmritaNambiar/Diachronic-Analysis-US-Court/blob/main/%5BScraping%5D_%5BPre_Processing%5D_ANLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installation
!pip install spacy
!python -m spacy download en_core_web_sm

# Core Python libraries
import os
import html
import re
import json
import time
from datetime import datetime
from typing import List, Dict, Optional, Tuple

# Data manipulation
import pandas as pd

# Web scraping
import requests

# Natural Language Processing
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.corpus import words as nltk_words
import spacy

# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('words')
nltk.download('averaged_perceptron_tagger_eng')

# Access Drive
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 46.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Data Scraper using CourtListner
Link: (https://www.courtlistener.com/)

### Periods Analyzed:
1. **Pre-Katz (1790-1967)**: ~842 cases
   - Legal context: Physical trespass doctrine
   - Landmark: Ex Parte Jackson (1878), Boyd v. United States (1886)
   - Ends with: Katz v. United States (1967)

2. **Post-Katz (1968-1985)**: ~5,140 cases
   - Legal context: Expectation of privacy framework
   - Milestone: Katz v. United States (1967) - establishes two-part test
   - Ends before: ECPA (Electronic Communications Privacy Act, 1986)

3. **Internet Regulation (1986-2000)**: ~8,011 cases
   - Legal context: Digital communications emergence
   - Milestone: ECPA (1986)
   - Period: Y2K and digital era emergence

In [ ]:
class FourthAmendmentScraper:

    PROGRESS_FILE = '/content/drive/My Drive/ANLP Final Project 25/preprocessed_data/raw_scraped/scraper_progress.json'
    OUTPUT_DIR = '/content/drive/My Drive/ANLP Final Project 25/preprocessed_data/raw_scraped/'

    def __init__(self, api_token: str):
        if not api_token:
            raise ValueError("API token required")

        self.api_token = api_token
        self.base_url = "https://www.courtlistener.com/api/rest/v4"
        self.headers = {
            "Authorization": f"Token {api_token}",
            "Content-Type": "application/json"
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)

        os.makedirs(self.OUTPUT_DIR, exist_ok=True)
        self.progress = self.load_progress()

    def load_progress(self) -> Dict:
        if os.path.exists(self.PROGRESS_FILE):
            try:
                with open(self.PROGRESS_FILE, 'r') as f:
                    return json.load(f)
            except:
                return {}
        return {}

    def save_progress(self, era_label: str, status: str, details: Dict = None) -> None:
        self.progress[era_label] = {
            "status": status,
            "timestamp": datetime.now().isoformat(),
            "details": details or {}
        }
        try:
            with open(self.PROGRESS_FILE, 'w') as f:
                json.dump(self.progress, f, indent=2)
        except:
            pass

    def validate_date_format(self, date_str: str) -> bool:
        try:
            datetime.fromisoformat(date_str)
            return True
        except ValueError:
            return False

    def search_cases(self,
                     date_range: Tuple[str, str],
                     era_label: str,
                     search_query: str = "Fourth Amendment",
                     max_attempts: int = 3) -> List[Dict]:
        start_date, end_date = date_range

        if not self.validate_date_format(start_date) or not self.validate_date_format(end_date):
            print(f"Invalid date range for {era_label}")
            return []

        cases = []
        cursor = None
        page_count = 0
        cases_by_year = {}

        print(f"\nSearching {era_label}: {start_date} to {end_date}")

        while True:
            page_count += 1
            attempt = 0

            while attempt < max_attempts:
                try:
                    params = {
                        'q': search_query,
                        'type': 'o',
                        'filed_after': start_date,
                        'filed_before': end_date,
                        'order_by': 'dateFiled asc'
                    }

                    if cursor:
                        params['cursor'] = cursor

                    response = self.session.get(
                        f"{self.base_url}/search/",
                        params=params,
                        timeout=30
                    )

                    if response.status_code == 429:
                        time.sleep(60)
                        continue

                    if response.status_code != 200:
                        attempt += 1
                        if attempt < max_attempts:
                            time.sleep(5)
                            continue
                        else:
                            break

                    data = response.json()
                    results = data.get('results', [])

                    if not results:
                        break

                    for result in results:
                        case_info = self.extract_case_info(result, start_date, end_date)
                        if case_info:
                            cases.append(case_info)
                            year = case_info['date_filed'][:4]
                            cases_by_year[year] = cases_by_year.get(year, 0) + 1

                    if page_count % 5 == 0:
                        print(f"  Page {page_count}: {len(cases)} cases found")
                        self.save_progress(
                            era_label,
                            'in_progress',
                            {
                                'cases_found': len(cases),
                                'pages_processed': page_count,
                                'cases_by_year': cases_by_year
                            }
                        )

                    next_url = data.get('next')
                    cursor = self.extract_cursor(next_url)

                    if not cursor:
                        break

                    time.sleep(1)
                    break

                except requests.exceptions.Timeout:
                    attempt += 1
                    time.sleep(5)
                except requests.exceptions.ConnectionError:
                    attempt += 1
                    time.sleep(10)
                except Exception as e:
                    attempt += 1
                    time.sleep(5)

            if attempt >= max_attempts and page_count > 1:
                break

        print(f"Found {len(cases)} cases for {era_label}")

        if cases_by_year:
            print(f"Distribution by year:")
            for year in sorted(cases_by_year.keys()):
                print(f"  {year}: {cases_by_year[year]} cases")

        return cases

    def extract_cursor(self, next_url: Optional[str]) -> Optional[str]:
        if not next_url:
            return None
        try:
            import urllib.parse as urlparse
            parsed = urlparse.urlparse(next_url)
            params = urlparse.parse_qs(parsed.query)
            return params.get('cursor', [None])[0]
        except:
            return None

    def extract_case_info(self, result: Dict, start_date: str, end_date: str) -> Optional[Dict]:
        try:
            # API returns both formats, try both
            date_filed = result.get('dateFiled') or result.get('date_filed')
            if not date_filed:
                return None

            try:
                filed_date = datetime.fromisoformat(date_filed)
                start_dt = datetime.fromisoformat(start_date)
                end_dt = datetime.fromisoformat(end_date)

                if filed_date < start_dt or filed_date > end_dt:
                    return None
            except ValueError:
                return None

            cluster_id = result.get('cluster_id')
            if not cluster_id:
                return None

            citations = result.get('citation', [])
            citation_str = ', '.join(citations) if citations else 'No citation'

            return {
                'cluster_id': cluster_id,
                'case_name': result.get('caseName') or result.get('case_name', 'Unknown Case'),
                'court': result.get('court', 'Unknown Court'),
                'court_id': result.get('court_id', ''),
                'citation': citation_str,
                'date_filed': date_filed,
                'docket_id': result.get('docket_id', ''),
                'snippet': result.get('snippet', ''),
                'absolute_url': result.get('absolute_url', ''),
                'opinions': result.get('opinions', [])
            }
        except Exception as e:
            return None

    def fetch_opinion_text(self, case: Dict) -> Optional[str]:
        try:
            opinions = case.get('opinions', [])
            if not opinions:
                snippet = case.get('snippet', '')
                if len(snippet) > 50:
                    return snippet
                return None

            full_text = []
            for opinion in opinions:
                opinion_id = opinion.get('id')
                if not opinion_id:
                    continue

                try:
                    response = self.session.get(
                        f"{self.base_url}/opinions/{opinion_id}/",
                        timeout=30
                    )
                    if response.status_code != 200:
                        continue

                    opinion_data = response.json()
                    text = (opinion_data.get('plain_text') or
                           opinion_data.get('html') or
                           opinion_data.get('text'))

                    if text:
                        if '<' in text and '>' in text:
                            text = self.clean_html(text)
                        full_text.append(text.strip())

                    time.sleep(0.3)
                except:
                    continue

            combined_text = ' '.join(full_text)
            if not combined_text or len(combined_text) < 50:
                snippet = case.get('snippet', '')
                if snippet and len(snippet) > 50:
                    return snippet
                return None

            return combined_text.strip()
        except Exception as e:
            return None

    def clean_html(self, html_text: str) -> str:
        text = re.sub(r'<[^>]+>', '', html_text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def chunk_text(self, text: str, chunk_size: int = 512) -> List[str]:
        words = text.split()
        chunks = []
        for i in range(0, len(words), chunk_size):
            chunk = ' '.join(words[i:i+chunk_size])
            chunks.append(chunk)
        return chunks

    def process_cases(self, cases: List[Dict], era_label: str) -> pd.DataFrame:
        all_chunks = []
        seen_case_ids = set()
        skipped_empty = 0
        skipped_duplicate = 0

        print(f"\nProcessing {len(cases)} cases for {era_label}")

        for idx, case in enumerate(cases, 1):
            try:
                case_id = case['cluster_id']
                case_name = case['case_name'][:50]
                date_filed = case['date_filed']

                if case_id in seen_case_ids:
                    skipped_duplicate += 1
                    continue

                if idx % 20 == 0:
                    print(f"  Processing case {idx}/{len(cases)}")

                full_text = self.fetch_opinion_text(case)
                if not full_text:
                    skipped_empty += 1
                    continue

                word_count = len(full_text.split())
                if word_count < 50:
                    skipped_empty += 1
                    continue

                seen_case_ids.add(case_id)
                chunks = self.chunk_text(full_text)

                for chunk_num, chunk_text in enumerate(chunks):
                    all_chunks.append({
                        'case_id': case_id,
                        'case_name': case['case_name'],
                        'court': case['court'],
                        'citation': case['citation'],
                        'date_filed': case['date_filed'],
                        'chunk_number': chunk_num,
                        'total_chunks': len(chunks),
                        'text_chunk': chunk_text
                    })

                if idx % 50 == 0:
                    temp_df = pd.DataFrame(all_chunks)
                    temp_filename = os.path.join(self.OUTPUT_DIR, f"temp_{era_label}.csv")
                    temp_df.to_csv(temp_filename, index=False, encoding='utf-8')
                    time.sleep(1)

            except Exception as e:
                continue

        print(f"Created {len(all_chunks)} chunks from {len(seen_case_ids)} unique cases")
        print(f"Skipped: {skipped_empty} empty, {skipped_duplicate} duplicates")

        return pd.DataFrame(all_chunks)

    def save_dataset(self, df: pd.DataFrame, filename: str, era_label: str) -> None:
        initial_rows = len(df)
        df = df.drop_duplicates(subset=['case_id', 'chunk_number'], keep='first')
        final_rows = len(df)

        if initial_rows != final_rows:
            print(f"Removed {initial_rows - final_rows} duplicate chunks")

        filepath = os.path.join(self.OUTPUT_DIR, filename)
        df.to_csv(filepath, index=False, encoding='utf-8')

        self.save_progress(
            era_label,
            'complete',
            {
                'total_rows': final_rows,
                'unique_cases': int(df['case_id'].nunique()),
                'filename': filename,
                'date_range': {
                    'min': str(df['date_filed'].min()),
                    'max': str(df['date_filed'].max())
                }
            }
        )

        print(f"\nSaved {era_label}: {filepath}")
        print(f"  Rows: {final_rows}, Unique cases: {int(df['case_id'].nunique())}")
        if len(df) > 0:
            print(f"  Date range: {df['date_filed'].min()} to {df['date_filed'].max()}")

        temp_filename = os.path.join(self.OUTPUT_DIR, f"temp_{era_label}.csv")
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

def run_scraper():

    # Please get a token key from www.courtlistener.com to perform scraping
    API_TOKEN = "YOUR API TOKEN KEY"

    PERIODS = [
        {
            'start': '1790-01-01',
            'end': '1967-12-31',
            'label': 'pre_katz',
            'description': 'Pre-Katz Era (1790-1967)'
        },
        {
            'start': '1968-01-01',
            'end': '1985-12-31',
            'label': 'post_katz',
            'description': 'Post-Katz Early Era (1968-1985)'
        },
        {
            'start': '1986-01-01',
            'end': '2000-12-31',
            'label': 'internet_regulation',
            'description': 'Internet Regulation Era (1986-2000)'
        }
    ]

    print("\nFourth Amendment Scraper - Three Period Analysis (1790-2000)")
    print("\nPeriods:")
    for i, period in enumerate(PERIODS, 1):
        print(f"  {i}. {period['label']}: {period['start']} to {period['end']}")

    scraper = FourthAmendmentScraper(API_TOKEN)

    for period in PERIODS:
        era_label = period['label']
        filename = f"{era_label}_fourth_amendment.csv"

        print(f"\n{'='*60}")
        print(f"Period: {period['description']}")
        print(f"{'='*60}")

        cases = scraper.search_cases(
            date_range=(period['start'], period['end']),
            era_label=era_label,
            search_query="Fourth Amendment"
        )

        if cases:
            scraper.save_progress(era_label, 'processing', {'cases_found': len(cases)})

            df = scraper.process_cases(cases, era_label)
            if not df.empty:
                scraper.save_dataset(df, filename, era_label)
            else:
                print(f"No valid data for {era_label}")
                scraper.save_progress(era_label, 'failed', {'reason': 'No valid data'})
        else:
            print(f"No cases found for {era_label}")
            scraper.save_progress(era_label, 'failed', {'reason': 'No cases found'})

        print("Waiting 10 seconds before next period...")
        time.sleep(10)

    print(f"\n{'='*60}")
    print("All periods complete")
    print(f"{'='*60}\n")

    print("Summary:")
    for period in PERIODS:
        era_label = period['label']
        filename = f"{era_label}_fourth_amendment.csv"
        filepath = os.path.join(scraper.OUTPUT_DIR, filename)

        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath)
                print(f"\n{era_label}:")
                print(f"  File: {filename}")
                print(f"  Cases: {int(df['case_id'].nunique())}")
                print(f"  Chunks: {len(df)}")
                if len(df) > 0:
                    print(f"  Date range: {df['date_filed'].min()} to {df['date_filed'].max()}")
            except Exception as e:
                print(f"\n{era_label}: Error reading file")
        else:
            print(f"\n{era_label}: File not found")

if __name__ == "__main__":
    run_scraper()


Fourth Amendment Scraper - Three Period Analysis (1790-2000)

Periods:
  1. pre_katz: 1790-01-01 to 1967-12-31
  2. post_katz: 1968-01-01 to 1985-12-31
  3. internet_regulation: 1986-01-01 to 2000-12-31

Period: Pre-Katz Era (1790-1967)

Searching pre_katz: 1790-01-01 to 1967-12-31
  Page 5: 100 cases found
  Page 10: 200 cases found
  Page 15: 300 cases found
  Page 20: 400 cases found


KeyboardInterrupt: 

## Process Dataframe for Pre-Processing

In [ ]:
# Loading individual files for pre-processing

file_path_pre_katz = "/content/drive/My Drive/ANLP Final Project 25/preprocessed_data/raw_scraped/pre_katz_fourth_amendment.csv"
file_path_post_katz = '/content/drive/My Drive/ANLP Final Project 25/preprocessed_data/raw_scraped/post_katz_fourth_amendment.csv'
file_path_internet_reg = '/content/drive/My Drive/ANLP Final Project 25/preprocessed_data/raw_scraped/internet_regulation_fourth_amendment.csv'

def process_dataframe(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return pd.DataFrame()

    temp_df = pd.read_csv(file_path)
    temp_df = temp_df.sort_values(by=['case_id', 'chunk_number'])
    processed_df = temp_df.groupby('case_id')['text_chunk'].apply(lambda x: ' '.join(x)).reset_index()
    processed_df.rename(columns={'text_chunk': 'combined_text_chunk'}, inplace=True)
    return processed_df

print("Loading scraped data...")
df_pre_katz = process_dataframe(file_path_pre_katz)
df_post_katz = process_dataframe(file_path_post_katz)
df_internet_reg = process_dataframe(file_path_internet_reg)

print(f"\nPre-Katz: {len(df_pre_katz)} cases")
print(f"Post-Katz: {len(df_post_katz)} cases")
print(f"Internet Regulation: {len(df_internet_reg)} cases")

if len(df_pre_katz) > 0:
    print("\nPre-Katz DataFrame:")
    display(df_pre_katz.head())
if len(df_post_katz) > 0:
    print("\nPost-Katz DataFrame:")
    display(df_post_katz.head())
if len(df_internet_reg) > 0:
    print("\nInternet Regulation DataFrame:")
    display(df_internet_reg.head())

Loading scraped data...

Pre-Katz: 842 cases
Post-Katz: 5140 cases
Internet Regulation: 8011 cases

Pre-Katz DataFrame:


,case_id,combined_text_chunk
0,89759,96 U.S. 727 24 L.Ed. 877 EX PARTE JACKSON. Oct...
1,91573,6 S.Ct. 524 116 U.S. 616 29 L.Ed. 746 BOYD and...
2,93234,142 U.S. 547 12 S.Ct. 195 35 L.Ed. 1110 COUNSE...
3,93665,149 U.S. 698 13 S.Ct. 1016 37 L.Ed. 905 FONG Y...
4,94399,161 U.S. 475 16 S.Ct. 641 40 L.Ed. 777 UNITED ...



Post-Katz DataFrame:


,case_id,combined_text_chunk
0,107577,389 U.S. 560 88 S.Ct. 660 19 L.Ed.2d 770 COMMO...
1,107625,390 U.S. 234 88 S.Ct. 992 19 L.Ed.2d 1067 Jame...
2,107636,390 U.S. 377 88 S.Ct. 967 19 L.Ed.2d 1247 Thom...
3,107662,390 U.S. 611 88 S.Ct. 1335 20 L.Ed.2d 182 John...
4,107684,391 U.S. 123 88 S.Ct. 1620 20 L.Ed.2d 476 Geor...



Internet Regulation DataFrame:


,case_id,combined_text_chunk
0,4751,UNITED STATES COURT OF APPEALS For the Fifth C...
1,4769,UNITED STATES COURT OF APPEALS FIFTH CIRCUIT _...
2,4773,IN THE UNITED STATES COURT OF APPEALS FOR THE ...
3,4787,UNITED STATES COURT OF APPEALS FOR THE FIFTH C...
4,4808,UNITED STATES COURT OF APPEALS FOR THE FIFTH C...


# Pre-Processsing
### Five-Layer Preprocessing Pipeline:

**Layer 1: Text Normalization**
- Resolve HTML entities
- Normalize unicode characters (em-dashes, curly quotes)
- Remove excess whitespace
- Impact: ~99% clean text

**Layer 2: Structural Cleanup (Citation Removal)**
- Remove citation patterns (U.S. Reporter, Supreme Court Reporter, Federal Reporter, Case numbers)
- Example: "96 U.S. 727" removed from text
- Rationale: Prevents citation tokens from dominating embeddings
- Caveat: Severs 2-5% of doctrinal references

**Layer 3: OCR Error Correction**
- Fix common OCR artifacts: "seiz ure" → "seizure", "ca riage" → "carriage"
- Regex-based pattern matching
- Catches ~95% of systematic errors, ~5% remain as noise

**Layer 4: Lemmatization & Tokenization**
- spaCy POS tagging
- WordNet lemmatization (reduces inflected forms)
- Example: "searched", "searching", "searches" → "search"
- Caveat: Loses tense/aspect (ongoing vs. completed action)

**Layer 5: Stopword Filtering**
- Remove high-frequency non-content words
- IMPORTANT: Retain negation words ("no", "not", "never")
- IMPORTANT: Retain contrast words ("but", "yet", "however")
- Remove: 170+ English stopwords + legal procedure terms

In [ ]:
# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    from spacy.cli import download
    download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

# Build vocabulary for heuristic merging
valid_vocab = set(nltk_words.words())

# Legal lexicon: high-value terms that must be preserved
legal_lexicon = {
    'seizure', 'seizures', 'seizure', 'seizures',
    'warrant', 'warrants', 'warranted',
    'search', 'searches',
    'unreasonable', 'reasonable',
    'property', 'private', 'privacy',
    'surveillance', 'eavesdrop', 'eavesdropping',
    'wiretap',
    'incrimination', 'self-incrimination',
    'arrest', 'arrest',
    'confiscation', 'forfeiture',
    'contraband',
    'magistrate', 'magistrates',
    'affidavit', 'affidavits',
    'habeas', 'corpus', 'habeas-corpus',
    'certiorari',
    'carriage',
    'pamphlet',
    'fraud',
    'probable-cause', 'probable cause',
    'search-warrant',
}

valid_vocab = valid_vocab.union(legal_lexicon)

# C. Latin Maxims
latin_phrases = {
    'habeas corpus': 'habeas_corpus',
    'certiorari': 'certiorari',
    'ex post facto': 'ex_post_facto',
    'stare decisis': 'stare_decisis',
    'prima facie': 'prima_facie',
    'amicus curiae': 'amicus_curiae',
    'in personam': 'in_personam',
    'in rem': 'in_rem'
}

# D. Custom Stopwords
stops = set(stopwords.words('english'))
critical_retention = {'no', 'not', 'nor', 'neither', 'never', 'none', 'against', 'versus'}
stops = stops - critical_retention

legal_noise = {
    'court', 'case', 'plaintiff', 'defendant', 'appellant', 'appellee',
    'petitioner', 'respondent', 'id', 'ibid', 'supra', 'et', 'al',
    'mr', 'mrs', 'messrs', 'honor', 'justice', 'argument', 'counsel'
}
stops.update(legal_noise)

# Global normalization and encoding repair
def normalize_text(text):
    if not isinstance(text, str):
        return ""

    # 1. HTML unescape
    text = html.unescape(text)

    # 2. Normalize unicode
    text = text.replace('\u2014', '-')
    text = text.replace('\u2013', '-')
    text = text.replace('\u201c', '"')
    text = text.replace('\u201d', '"')

    # 3. Collapse multiple spaces/newlines
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Layer 2: Remove procedural debris and citation noise.
def remove_legal_citations_and_noise(text):

    if not isinstance(text, str):
        return ""

    # Remove U.S. Supreme Court citations (e.g., "338 U.S. 25")
    text = re.sub(r'\b\d{1,3}\s+U\.?S\.?\s+\d{1,4}\b', ' ', text)
    text = re.sub(r'\b\d{1,3}\s+S\.?Ct\.?\s+\d{1,4}\b', ' ', text)
    text = re.sub(r'\b\d{1,3}\s+L\.?Ed\.?(?:2d)?\s+\d{1,4}\b', ' ', text)
    text = re.sub(r'\b\d{1,3}\s+F\.?(?:2d|3d)\s+\d{1,4}\b', ' ', text)
    text = re.sub(r'\b\d{1,3}\s+F\.?Supp\.?(?:2d|3d)?\s+\d{1,4}\b', ' ', text)

    # Remove section markers (§, §§)
    text = re.sub(r'§+\s*\d+[a-zA-Z0-9\-]*', ' ', text)

    # Remove case number headers
    text = re.sub(r'\bNo\.?\s+\d+[-\dA-Za-z]*\b', ' ', text, flags=re.IGNORECASE)

    # Remove "Argument of Counsel..." and similar bracketed procedural notes
    text = re.sub(r'\bArgument of Counsel.*?(?:\.|omitted)', ' ', text, flags=re.IGNORECASE)

    # Remove "Mr. Justice X delivered the opinion" headers
    text = re.sub(r'Mr\.?\s+Justice\s+\w+\s+delivered the opinion', ' ', text, flags=re.IGNORECASE)

    text = re.sub(r'-{3,}PAGE-{3,}', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


In [ ]:
# Comprehensive OCR patterns fix
def aggressive_ocr_correction(text):
    if not isinstance(text, str):
        return ""



    # Most common OCR issue - line breaks in middle of words
    space_separated_fixes = [
        (r'\bp\s+wer\b', 'power'),
        (r'\bcar\s+iage\b', 'carriage'),
        (r'\bwou\s+d\b', 'would'),
        (r'\btresp\s+ass\b', 'trespass'),
        (r'\bcon\s+stitution\b', 'constitution'),
        (r'\bseiz\s*-\s*ure\b', 'seizure'),
        (r'\bseiz\s+ure\b', 'seizure'),
        (r'\bseiz\s+ures\b', 'seizures'),
        (r'\bintru\s*-\s*sion\b', 'intrusion'),
        (r'\bintru\s+sion\b', 'intrusion'),
        (r'\bun\s+reasonable\b', 'unreasonable'),
    ]

    # Truncation error
    truncation_fixes = [
        (r'\bfrand\b', 'fraud'),
        (r'\bpamphle\b', 'pamphlet'),
        (r'\bintercep\b', 'interception'),
        (r'\bprevacy\b', 'privacy'),
        (r'\bexpecta\b', 'expectation'),
        (r'\bseizuer\b', 'seizure'),
    ]

    # Spaces removed between words
    concatenation_fixes = [
        (r'\bproducethem\b', 'produce them'),
        (r'\bfourthamendment\b', 'fourth amendment'),
        (r'\bofprivacy\b', 'of privacy'),
    ]

    # Multiple forms → single standard
    compound_fixes = [
        (r'\bwire\s+tapping\b', 'wiretapping'),
        (r'\bwire\s*-\s*tapping\b', 'wiretapping'),
    ]

    # SPECIAL CASES
    special_cases = [
        (r'\blas\b', 'law'),
        (r'\b(?:as|naw|lav)\s+(?=to\s+be)', 'law'),
    ]


    all_corrections = (
        space_separated_fixes +
        truncation_fixes +
        concatenation_fixes +
        compound_fixes +
        special_cases
    )

    # Apply all corrections
    for pattern, replacement in all_corrections:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text


In [ ]:
# Fix seiz-ure, seiz ure, car iage using dictionary lookahead
def fix_broken_words_heuristic(text):
    if not isinstance(text, str):
        return ""

    def replace_hyphen(match):
        p1, p2 = match.group(1), match.group(2)
        combined = (p1 + p2).lower()

        if combined in valid_vocab:
            if p1.lower() not in valid_vocab or p2.lower() not in valid_vocab:
                return p1 + p2

        return match.group(0)

    def replace_space(match):
        p1, p2 = match.group(1), match.group(2)
        combined = (p1 + p2).lower()

        # Same logic for space-separated pairs
        if combined in valid_vocab:
            if p1.lower() not in valid_vocab or p2.lower() not in valid_vocab:
                return p1 + p2

        return match.group(0)

    # Pattern A: Hyphenated breaks (e.g., "seiz-ure", "un- reasonable")
    text = re.sub(r'\b([a-zA-Z]{2,})\s*-\s*([a-zA-Z]{2,})\b', replace_hyphen, text)

    # Pattern B: Space breaks (e.g., "seiz ure", "car iage")
    text = re.sub(r'\b([a-zA-Z]{2,4})\s+([a-zA-Z]{2,4})\b', replace_space, text)

    return text


In [ ]:
# Clean tokenization using spaCy.
def lemmatize_and_tokenize(text):
    if not isinstance(text, str):
        return []

    doc = nlp(text)

    tokens = [
        token.lemma_.lower()
        for token in doc
        if (not token.is_stop
            and not token.is_punct
            and not token.like_num
            and token.is_alpha
            and len(token.text) > 2)
    ]

    return tokens

# Custom lemma normalization for legal verbs
LEGAL_VERB_NORMALIZER = {
    "wiretapping": "wiretap",
    "wiretapped": "wiretap",
    "wiretaps": "wiretap",
    "tapping": "tap",
    "tapped": "tap",
    "taps": "tap",
    "interceptions": "interception",
    "intercepted": "intercept",
    "intercepting": "intercept",
}

def normalize_legal_verbs(tokens):
    return [LEGAL_VERB_NORMALIZER.get(tok, tok) for tok in tokens]

# More terms to normalise
def normalize_compounds(text):
    text = re.sub(r"\bthird\s*-\s*party\b", "third_party", text, flags=re.IGNORECASE)
    text = re.sub(r"\bthird\s+party\b", "third_party", text, flags=re.IGNORECASE)
    text = re.sub(r"\bthird\s*part[y]?\b", "third_party", text, flags=re.IGNORECASE)

    return text

# Orchestrate all cleaning layers in order
def master_preprocessing_pipeline(text):
    text = normalize_text(text)
    text = remove_legal_citations_and_noise(text)
    text = aggressive_ocr_correction(text)
    text = normalize_compounds(text)
    text = fix_broken_words_heuristic(text)
    tokens = lemmatize_and_tokenize(text)
    tokens = normalize_legal_verbs(tokens)

    return tokens

In [ ]:
PREPROCESS_OUTPUT_DIR = '/content/drive/My Drive/ANLP Final Project 25/preprocessed_data/'
os.makedirs(PREPROCESS_OUTPUT_DIR, exist_ok=True)

dataframes_and_labels = [
    (df_pre_katz, 'pre_katz'),
    (df_post_katz, 'post_katz'),
    (df_internet_reg, 'internet_reg')
]

preprocessing_results = {}

print("=" * 80)
print("PREPROCESSING: CHECK AND LOAD OR PREPROCESS")
print("=" * 80)

files_to_process = []
files_to_load = []

for df, label in dataframes_and_labels:
    output_path = os.path.join(PREPROCESS_OUTPUT_DIR, f'{label}_preprocessed.csv')

    if os.path.exists(output_path):
        file_size_mb = os.path.getsize(output_path) / 1024 / 1024
        files_to_load.append((label, output_path, file_size_mb))
        print(f"[{label}] Found: {file_size_mb:.1f} MB")
    else:
        files_to_process.append((df, label))
        print(f"[{label}] Not found - will preprocess")

if files_to_process:
    print("\n" + "=" * 80)
    print("PREPROCESSING MISSING SLICES")
    print("=" * 80)

    for df, label in files_to_process:
        print(f"\n[{label}] Processing {len(df)} cases...")

        df['tokenized_text'] = df['combined_text_chunk'].astype(str).apply(
            master_preprocessing_pipeline
        )

        df['processed_text'] = df['tokenized_text'].apply(lambda x: ' '.join(x))

        essential_cols = ['case_id', 'case_name', 'court', 'citation', 'date_filed']
        output_cols = [col for col in essential_cols if col in df.columns]
        output_cols.extend(['processed_text', 'tokenized_text'])

        df_to_save = df[output_cols].copy()
        output_path = os.path.join(PREPROCESS_OUTPUT_DIR, f'{label}_preprocessed.csv')
        df_to_save.to_csv(output_path, index=False)

        file_size_mb = os.path.getsize(output_path) / 1024 / 1024

        preprocessing_results[label] = {
            'status': 'PREPROCESSED',
            'rows': len(df_to_save),
            'size_mb': file_size_mb,
            'path': output_path
        }

        print(f"Saved: {len(df_to_save)} rows, {file_size_mb:.1f} MB")

if files_to_load:
    print("\n" + "=" * 80)
    print("LOADING EXISTING FILES")
    print("=" * 80)

    for label, output_path, file_size_mb in files_to_load:
        print(f"[{label}] Loading...")

        df_loaded = pd.read_csv(output_path)

        df_loaded['tokenized_text'] = df_loaded['tokenized_text'].apply(
            lambda x: eval(x) if isinstance(x, str) and x.startswith('[') else x
        )

        for i, (df, label_check) in enumerate(dataframes_and_labels):
            if label_check == label:
                dataframes_and_labels[i] = (df_loaded, label)
                break

        preprocessing_results[label] = {
            'status': 'LOADED',
            'rows': len(df_loaded),
            'size_mb': file_size_mb,
            'path': output_path
        }

        print(f"Loaded: {len(df_loaded)} rows, {file_size_mb:.1f} MB")

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

for label, results in preprocessing_results.items():
    status_icon = "✓" if results['status'] == 'LOADED' else "◈"
    print(f"{status_icon} {label:<20} | {results['rows']:>6} rows | {results['size_mb']:>7.1f} MB")

total_size = sum(r['size_mb'] for r in preprocessing_results.values())
loaded_count = sum(1 for r in preprocessing_results.values() if r['status'] == 'LOADED')
preprocessed_count = sum(1 for r in preprocessing_results.values() if r['status'] == 'PREPROCESSED')

print(f"\nLoaded: {loaded_count} | Preprocessed: {preprocessed_count} | Total: {total_size:.1f} MB")
print("=" * 80)

PREPROCESSING: CHECK AND LOAD OR PREPROCESS
[pre_katz] Found: 27.4 MB
[post_katz] Found: 188.2 MB
[internet_reg] Found: 313.8 MB

LOADING EXISTING FILES
[pre_katz] Loading...
Loaded: 842 rows, 27.4 MB
[post_katz] Loading...
Loaded: 5140 rows, 188.2 MB
[internet_reg] Loading...
Loaded: 8011 rows, 313.8 MB

SUMMARY
✓ pre_katz             |    842 rows |    27.4 MB
✓ post_katz            |   5140 rows |   188.2 MB
✓ internet_reg         |   8011 rows |   313.8 MB

Loaded: 3 | Preprocessed: 0 | Total: 529.4 MB


#### Pre-Processing Complete...

## Loading the pre-processed dataset for testing

In [ ]:
# Update the path to your Google Drive location
PREPROCESS_OUTPUT_DIR = '/content/drive/My Drive/ANLP Final Project 25/preprocessed_data/'

def load_all_slices():
    # Load all preprocessed slices from CSV
    slice_labels = ['pre_katz', 'post_katz', 'internet_reg']

    loaded_dfs = {}

    for label in slice_labels:
        input_path = os.path.join(PREPROCESS_OUTPUT_DIR, f'{label}_preprocessed.csv')

        if os.path.exists(input_path):
            df = pd.read_csv(input_path)

            # Convert string representation of lists back to actual lists
            if 'tokenized_text' in df.columns:
                df['tokenized_text'] = df['tokenized_text'].apply(
                    lambda x: eval(x) if isinstance(x, str) and x.startswith('[') else x
                )

            loaded_dfs[label] = df
            print(f" {label:<20} | {len(df):>6} rows")
        else:
            print(f" {label:<20} | FILE NOT FOUND at {input_path}")

    return loaded_dfs

# Load all preprocessed data
print("Loading preprocessed data from Google Drive...")
print(f"Looking in: {PREPROCESS_OUTPUT_DIR}\n")

loaded_dfs = load_all_slices()

# Assign to individual dataframe variables
df_pre_katz = loaded_dfs.get('pre_katz', pd.DataFrame())
df_post_katz = loaded_dfs.get('post_katz', pd.DataFrame())
df_internet_reg = loaded_dfs.get('internet_reg', pd.DataFrame())

print("\n All dataframes loaded from Google Drive checkpoint")

# Display summary
print("\n" + "="*50)
print("LOADED DATAFRAMES SUMMARY")
print("="*50)
for name, df in [('df_pre_katz', df_pre_katz),
                  ('df_post_katz', df_post_katz),
                  ('df_internet_reg', df_internet_reg)]:
    if not df.empty:
        print(f"{name:<20} : {len(df)} rows, {len(df.columns)} columns")
    else:
        print(f"{name:<20} : EMPTY")

Loading preprocessed data from Google Drive...
Looking in: /content/drive/My Drive/ANLP Final Project 25/preprocessed_data/

 pre_katz             |    842 rows
 post_katz            |   5140 rows
 internet_reg         |   8011 rows

 All dataframes loaded from Google Drive checkpoint

LOADED DATAFRAMES SUMMARY
df_pre_katz          : 842 rows, 3 columns
df_post_katz         : 5140 rows, 3 columns
df_internet_reg      : 8011 rows, 3 columns


In [ ]:
print("Pre-Katz DataFrame with cleaned text:")
display(df_pre_katz.head())

Pre-Katz DataFrame with cleaned text:


,case_id,processed_text,tokenized_text
0,89759,parte jackson october term petition writ habea...,"[parte, jackson, october, term, petition, writ..."
1,91573,boyd claimants etc united states file february...,"[boyd, claimants, etc, united, states, file, f..."
2,93234,counselman hitchcock marshal january november ...,"[counselman, hitchcock, marshal, january, nove..."
3,93665,fong yue ting united states wong quan lee joe ...,"[fong, yue, ting, united, states, wong, quan, ..."
4,94399,united states zucker march whitney asst atty u...,"[united, states, zucker, march, whitney, asst,..."


In [ ]:
print("\nPost-Katz DataFrame with cleaned text:")
display(df_post_katz.head())


Post-Katz DataFrame with cleaned text:


,case_id,processed_text,tokenized_text
0,107577,commonwealth massachusetts petitioner donald p...,"[commonwealth, massachusetts, petitioner, dona..."
1,107625,james harris petitioner united states argue de...,"[james, harris, petitioner, united, states, ar..."
2,107636,thomas earl simmons petitioners united states ...,"[thomas, earl, simmons, petitioners, united, s..."
3,107662,john earl cameron appellants paul johnson etc ...,"[john, earl, cameron, appellants, paul, johnso..."
4,107684,george william bruton petitioner united states...,"[george, william, bruton, petitioner, united, ..."


In [ ]:
print("\nInternet Regulation DataFrame with cleaned text:")
display(df_internet_reg.head())


Internet Regulation DataFrame with cleaned text:


,case_id,processed_text,tokenized_text
0,4751,united states court appeals circuit united sta...,"[united, states, court, appeals, circuit, unit..."
1,4769,united states court appeals circuit united sta...,"[united, states, court, appeals, circuit, unit..."
2,4773,united states court appeals circuit united sta...,"[united, states, court, appeals, circuit, unit..."
3,4787,united states court appeals circuit united sta...,"[united, states, court, appeals, circuit, unit..."
4,4808,united states court appeals circuit united sta...,"[united, states, court, appeals, circuit, unit..."
